<a id="rotation-2d"></a>

$$
\text{2D 旋转 (Rotation): 由和角公式推导} \\
\text{点用极坐标表示 } p = (x, y) = (r \cos\varphi, r \sin\varphi) \text{, 逆时针旋转 } \theta \text{ 后角度变成 } \varphi + \theta \text{, 长度不变} \\
x' = r \cos(\varphi + \theta) = r \cos\varphi \cos\theta - r \sin\varphi \sin\theta = x \cos\theta - y \sin\theta \\
y' = r \sin(\varphi + \theta) = r \sin\varphi \cos\theta + r \cos\varphi \sin\theta = x \sin\theta + y \cos\theta \\
\begin{bmatrix} x' \\ y' \end{bmatrix} = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix} \begin{bmatrix} x \\ y \end{bmatrix} \\ \quad \\
\text{复数的视角: 把点看成 } z = x + iy = r e^{i\varphi} \text{, 旋转就是乘以 } e^{i\theta} \text{ (欧拉公式)} \\
z' = r e^{i(\varphi + \theta)} = z \, e^{i\theta} = (x + iy)(\cos\theta + i \sin\theta) = (x\cos\theta - y\sin\theta) + i(x\sin\theta + y\cos\theta) \\
\text{和矩阵完全一样. 四元数就是把这个思想推广到 3D} \\ \quad \\
\text{性质: } R^T R = I \text{ (正交矩阵), } \det R = 1 \text{, 保持长度和夹角; } R(\alpha) R(\beta) = R(\alpha + \beta) \text{, 2D 旋转可以交换顺序}
$$

数学基础: [欧拉公式 (泰勒展开)](../math_basics/calculus.ipynb#taylor) · [正交矩阵](../math_basics/linear_algebra.ipynb#orthogonal) · [行列式](../math_basics/linear_algebra.ipynb#determinant)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation, Slerp

np.set_printoptions(suppress=True, precision=4)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

def R2(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

theta = np.deg2rad(35)
pts = np.random.default_rng(0).normal(size=(5, 2))
by_matrix = pts @ R2(theta).T
by_complex = (pts[:, 0] + 1j * pts[:, 1]) * np.exp(1j * theta)
print(f'矩阵旋转 == 复数乘以 e^(i theta) ? {np.allclose(by_matrix, np.column_stack([by_complex.real, by_complex.imag]))}')
print(f'R^T R == I ? {np.allclose(R2(theta).T @ R2(theta), np.eye(2))}, det R = {np.linalg.det(R2(theta)):.6f}')
print(f'长度不变 ? {np.allclose(np.linalg.norm(pts, axis=1), np.linalg.norm(by_matrix, axis=1))}')
print(f'R(a) R(b) == R(a + b) ? {np.allclose(R2(0.3) @ R2(1.1), R2(1.4))}, 可交换 ? {np.allclose(R2(0.3) @ R2(1.1), R2(1.1) @ R2(0.3))}')

<a id="rotation-3d"></a>

$$
\text{3D 旋转: 绕坐标轴的旋转矩阵 (推导见 homogeneous\_transforms)} \\
R_x(\theta) = \begin{bmatrix} 1 & 0 & 0 \\ 0 & \cos\theta & -\sin\theta \\ 0 & \sin\theta & \cos\theta \end{bmatrix}, \
R_y(\theta) = \begin{bmatrix} \cos\theta & 0 & \sin\theta \\ 0 & 1 & 0 \\ -\sin\theta & 0 & \cos\theta \end{bmatrix}, \
R_z(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta & 0 \\ \sin\theta & \cos\theta & 0 \\ 0 & 0 & 1 \end{bmatrix} \\ \quad \\
\text{3D 旋转矩阵的特征:} \\
1. \ R^T R = I \text{: 三列是两两垂直的单位向量 (三个坐标轴转到的位置)} \\
2. \ \det R = +1 \text{ (det = -1 是反射, 会把右手系变成左手系)} \\
3. \ \text{3D 旋转一般不能交换顺序: } R_x R_y \neq R_y R_x \\
4. \ \text{有一个特征值 = 1, 对应的特征向量就是旋转轴 (轴上的点不动: } R k = k \text{)}
$$

数学基础: [正交矩阵](../math_basics/linear_algebra.ipynb#orthogonal) · [特征值与特征向量](../math_basics/linear_algebra.ipynb#eigen)

In [ ]:
def Rx(t):
    c, s = np.cos(t), np.sin(t)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])

def Ry(t):
    c, s = np.cos(t), np.sin(t)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])

def Rz(t):
    c, s = np.cos(t), np.sin(t)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

R_a = Rx(np.pi / 2) @ Ry(np.pi / 2)
R_b = Ry(np.pi / 2) @ Rx(np.pi / 2)
print(f'Rx Ry == Ry Rx ? {np.allclose(R_a, R_b)}')
print(f'Rx(90) Ry(90) 作用在 x 轴上 = {R_a @ [1, 0, 0]}, Ry(90) Rx(90) 作用在 x 轴上 = {R_b @ [1, 0, 0]}')

R_any = Rz(0.3) @ Ry(-1.1) @ Rx(2.0)
print(f'R^T R == I ? {np.allclose(R_any.T @ R_any, np.eye(3))}, det = {np.linalg.det(R_any):.6f}')
eigval, eigvec = np.linalg.eig(R_any)
k_axis = np.real(eigvec[:, np.argmin(np.abs(eigval - 1))])
print(f'特征值 = {np.round(eigval, 4)}')
print(f'旋转轴 (特征值 1 的特征向量) = {k_axis}, R k == k ? {np.allclose(R_any @ k_axis, k_axis)}')
print(f'scipy 的旋转向量方向 = {Rotation.from_matrix(R_any).as_rotvec() / np.linalg.norm(Rotation.from_matrix(R_any).as_rotvec())} (和轴平行, 可能差一个符号)')

<a id="rodrigues"></a>

$$
\text{罗德里格斯旋转公式 (Rodrigues' Rotation Formula): 绕任意单位轴 k 旋转 } \theta \\ \quad \\
\text{1. 把 v 分解成平行于 k 和垂直于 k 的两部分 (投影):} \\
v_\parallel = (k \cdot v) \, k, \quad v_\perp = v - (k \cdot v) \, k \\
\text{2. 平行部分在轴上, 旋转后不变: } v_\parallel' = v_\parallel \\
\text{3. 垂直部分在垂直于 k 的平面里做 2D 旋转. 这个平面里需要两个互相垂直, 长度相同的基向量:} \\
\quad v_\perp \text{ 和 } k \times v \text{ (叉积垂直于 k 和 v; 并且 } k \times v = k \times v_\perp \text{, 长度 } = |k||v_\perp|\sin 90° = |v_\perp| \text{)} \\
\quad \text{套用 2D 旋转: } v_\perp' = \cos\theta \, v_\perp + \sin\theta \, (k \times v) \\
\text{4. 合起来:} \\
v' = v_\parallel + \cos\theta \, v_\perp + \sin\theta \, (k \times v) = (k \cdot v) k + \cos\theta \, (v - (k \cdot v) k) + \sin\theta \, (k \times v) \\
v' = v \cos\theta + (k \times v) \sin\theta + k (k \cdot v)(1 - \cos\theta) \\ \quad \\
\text{矩阵形式: 叉积可以写成矩阵乘法 } k \times v = K v, \quad K = \begin{bmatrix} 0 & -k_z & k_y \\ k_z & 0 & -k_x \\ -k_y & k_x & 0 \end{bmatrix} \text{ (反对称矩阵)} \\
K^2 v = k \times (k \times v) = k (k \cdot v) - v (k \cdot k) = k(k \cdot v) - v \quad (\text{二重叉积公式, } |k| = 1) \\
\text{把 } k(k \cdot v) = v + K^2 v \text{ 代入: } v' = v \cos\theta + \sin\theta \, K v + (1 - \cos\theta)(v + K^2 v) \\
R = I + \sin\theta \, K + (1 - \cos\theta) K^2
$$

数学基础: [投影 (向量在某方向上的分量)](../math_basics/linear_algebra.ipynb#projection)

In [ ]:
def cross_matrix(k):
    kx, ky, kz = k
    return np.array([[0, -kz, ky], [kz, 0, -kx], [-ky, kx, 0]])

def rodrigues_vector(v, k, theta):
    k = k / np.linalg.norm(k)
    return v * np.cos(theta) + np.cross(k, v) * np.sin(theta) + k * (k @ v) * (1 - np.cos(theta))

def rodrigues_matrix(k, theta):
    k = k / np.linalg.norm(k)
    K = cross_matrix(k)
    return np.eye(3) + np.sin(theta) * K + (1 - np.cos(theta)) * K @ K

k = np.array([1.0, 2.0, 2.0])
theta = 1.3
v = np.array([0.5, -1.0, 3.0])
R_rod = rodrigues_matrix(k, theta)
R_scipy = Rotation.from_rotvec(theta * k / np.linalg.norm(k)).as_matrix()
print(f'K v == k x v ? {np.allclose(cross_matrix(k) @ v, np.cross(k, v))}')
print(f'向量公式 == 矩阵公式 ? {np.allclose(rodrigues_vector(v, k, theta), R_rod @ v)}')
print(f'Rodrigues 矩阵 == scipy from_rotvec ? {np.allclose(R_rod, R_scipy)}')
print(f'绕 z 轴时 == Rz ? {np.allclose(rodrigues_matrix(np.array([0, 0, 1.0]), theta), Rz(theta))}')
k_unit = k / np.linalg.norm(k)
print(f'平行分量不变: k.v = {k_unit @ v:.4f}, k.(Rv) = {k_unit @ (R_rod @ v):.4f}')

<a id="gimbal-lock"></a>

$$
\text{欧拉角 (Euler Angles) 与万向锁 (Gimbal Lock)} \\
\text{用三个角表示旋转, 例如 } R(\alpha, \beta, \gamma) = R_z(\alpha) R_y(\beta) R_x(\gamma) \text{ (yaw, pitch, roll)} \\ \quad \\
\text{万向锁: 当 } \beta = 90° \text{ 时, 先引理: } R \, R_x(\gamma) \, R^{-1} \text{ 是绕 } R \, e_x \text{ 轴转 } \gamma \\
\text{(先用 } R^{-1} \text{ 把轴 } R e_x \text{ 转回 x 轴, 绕 x 轴转, 再转回去)} \\
R_y(90°) \, e_x = (0, 0, -1) = -e_z \implies R_y(90°) R_x(\gamma) R_y(90°)^{-1} = R_{-z}(\gamma) = R_z(-\gamma) \\
\implies R_y(90°) R_x(\gamma) = R_z(-\gamma) R_y(90°) \\
R(\alpha, 90°, \gamma) = R_z(\alpha) R_z(-\gamma) R_y(90°) = R_z(\alpha - \gamma) R_y(90°) \\
\text{结果只依赖 } \alpha - \gamma \text{: 改变 } \alpha \text{ 和改变 } \gamma \text{ 效果相同, 三个角只剩两个自由度} \\ \quad \\
\text{用 Jacobian 来看: 映射 } (\alpha, \beta, \gamma) \to R \text{ (9 个元素) 的 Jacobian 是 9x3 矩阵} \\
\text{一般情况下秩为 3; 在 } \beta = 90° \text{ 时 } \frac{\partial R}{\partial \alpha} = -\frac{\partial R}{\partial \gamma} \text{, 两列线性相关, 秩降为 2} \\
\text{(这时有一个方向的旋转无论怎么改欧拉角都做不到, 动画插值会出现突然的翻转)}
$$

数学基础: [Jacobian 矩阵](../math_basics/calculus.ipynb#jacobian) · [矩阵的秩](../math_basics/linear_algebra.ipynb#rank) · [SVD (用奇异值判断秩)](../math_basics/linear_algebra.ipynb#svd)

In [ ]:
def euler_zyx(a, b, g):
    return Rz(a) @ Ry(b) @ Rx(g)

def euler_jacobian(angles, h=1e-6):
    J = np.zeros((9, 3))
    for j in range(3):
        e = np.zeros(3); e[j] = h
        J[:, j] = (euler_zyx(*(angles + e)) - euler_zyx(*(angles - e))).ravel() / (2 * h)
    return J

print(f'和 scipy 一致 ? {np.allclose(euler_zyx(0.3, 0.5, -0.2), Rotation.from_euler("ZYX", [0.3, 0.5, -0.2]).as_matrix())}')
print(f'R_y(90) R_x(g) == R_z(-g) R_y(90) ? {np.allclose(Ry(np.pi / 2) @ Rx(0.7), Rz(-0.7) @ Ry(np.pi / 2))}')

a, g = 0.4, 0.1
for delta in [0.3, 1.0]:
    same = np.allclose(euler_zyx(a + delta, np.pi / 2, g + delta), euler_zyx(a, np.pi / 2, g))
    print(f'beta = 90 deg: alpha 和 gamma 同时加 {delta} 以后旋转矩阵不变 ? {same}')

for beta in [0.0, 0.8, 1.4, 1.55, np.pi / 2]:
    sv = np.linalg.svd(euler_jacobian(np.array([0.4, beta, 0.1])), compute_uv=False)
    print(f'beta = {np.rad2deg(beta):6.2f} deg: Jacobian 奇异值 = {sv}, 秩 = {np.sum(sv > 1e-6)}')

betas = np.linspace(0, np.pi / 2, 200)
min_sv = [np.linalg.svd(euler_jacobian(np.array([0.4, b, 0.1])), compute_uv=False)[-1] for b in betas]
plt.figure(figsize=(4, 3))
plt.plot(np.rad2deg(betas), min_sv, label='smallest singular value')
plt.xlabel('pitch beta (deg)')
plt.title('gimbal lock at beta = 90 deg')
plot_nicely()
plt.show()

<a id="quaternion"></a>

$$
\text{四元数 (Quaternion): } q = w + x i + y j + z k = (s, \vec{v}) \text{, 标量部分 s = w, 向量部分 } \vec{v} = (x, y, z) \\
\text{规则 (Hamilton): } i^2 = j^2 = k^2 = ijk = -1 \implies ij = k, \ jk = i, \ ki = j, \ ji = -k, \ kj = -i, \ ik = -j \\ \quad \\
\text{乘法公式推导: 先看两个纯四元数 (标量为 0) 相乘, 按分配律展开} \\
(a_1 i + a_2 j + a_3 k)(b_1 i + b_2 j + b_3 k) \\
= \underbrace{-(a_1 b_1 + a_2 b_2 + a_3 b_3)}_{i^2 = j^2 = k^2 = -1} + \underbrace{(a_2 b_3 - a_3 b_2) i + (a_3 b_1 - a_1 b_3) j + (a_1 b_2 - a_2 b_1) k}_{ij = k, ji = -k \cdots} \\
\implies \vec{a} \, \vec{b} = -\vec{a} \cdot \vec{b} + \vec{a} \times \vec{b} \\
\text{一般情况再加上标量部分 (标量和谁都可以交换):} \\
(s_1, \vec{v}_1)(s_2, \vec{v}_2) = (s_1 s_2 - \vec{v}_1 \cdot \vec{v}_2, \ s_1 \vec{v}_2 + s_2 \vec{v}_1 + \vec{v}_1 \times \vec{v}_2) \\
\text{因为有叉积, 四元数乘法不能交换} \\ \quad \\
\text{共轭 (Conjugate): } q^* = (s, -\vec{v}) \text{, } q q^* = (s^2 + |\vec{v}|^2, \vec{0}) = |q|^2 \text{, 单位四元数的逆 } q^{-1} = q^*
$$

<a id="quaternion-rotation"></a>

$$
\text{用单位四元数旋转: } q = (\cos\frac{\theta}{2}, \ \sin\frac{\theta}{2} \, \hat{k}) \text{, 把向量 v 看成纯四元数 } (0, \vec{v}) \\
v' = q \, v \, q^* \\ \quad \\
\text{推导 (为什么是半角): 把 v 分解成 } v_\parallel \text{ (平行于 } \hat{k} \text{) 和 } v_\perp \text{ (垂直于 } \hat{k} \text{)} \\
1. \ v_\parallel = c \hat{k} \text{: } \hat{k} \, v_\parallel = -c + \hat{k} \times c\hat{k} = -c = v_\parallel \hat{k} \text{, 所以 } v_\parallel \text{ 和 q 可以交换} \\
\quad q \, v_\parallel \, q^* = v_\parallel \, q \, q^* = v_\parallel \quad (\text{轴上的分量不动}) \\
2. \ v_\perp \text{: } \hat{k} \cdot v_\perp = 0 \text{, 所以 } \hat{k} v_\perp = \hat{k} \times v_\perp, \ v_\perp \hat{k} = v_\perp \times \hat{k} = -\hat{k} v_\perp \text{ (反交换)} \\
\quad v_\perp q^* = v_\perp (\cos\tfrac{\theta}{2} - \sin\tfrac{\theta}{2} \hat{k}) = (\cos\tfrac{\theta}{2} + \sin\tfrac{\theta}{2} \hat{k}) v_\perp = q \, v_\perp \\
\quad q \, v_\perp \, q^* = q^2 v_\perp, \quad q^2 = (\cos\theta, \sin\theta \, \hat{k}) \ (\text{倍角公式}) \\
\quad q^2 v_\perp = \cos\theta \, v_\perp + \sin\theta \, \hat{k} v_\perp = \cos\theta \, v_\perp + \sin\theta \, (\hat{k} \times v_\perp) \\
\text{和 Rodrigues 公式完全一样. q 在左右各乘一次, 每次贡献 } \frac{\theta}{2} \text{, 合起来转了 } \theta \\ \quad \\
\text{推论:} \\
1. \ q \text{ 和 } -q \text{ 表示同一个旋转 (} (-q) v (-q)^* = q v q^* \text{, 双重覆盖 double cover)} \\
2. \ \text{先 } q_1 \text{ 再 } q_2 \text{: } q_2 (q_1 v q_1^*) q_2^* = (q_2 q_1) v (q_2 q_1)^* \text{, 组合就是四元数相乘 (用到 } (ab)^* = b^* a^* \text{)} \\
3. \ \text{转成矩阵: 对 } e_x, e_y, e_z \text{ 分别算 } q e q^* \text{, 得到矩阵的三列, 展开整理得} \\
R(q) = \begin{bmatrix}
1 - 2(y^2 + z^2) & 2(xy - wz) & 2(xz + wy) \\
2(xy + wz) & 1 - 2(x^2 + z^2) & 2(yz - wx) \\
2(xz - wy) & 2(yz + wx) & 1 - 2(x^2 + y^2)
\end{bmatrix}
$$

In [ ]:
def qmul(q1, q2):
    s1, v1 = q1[0], q1[1:]
    s2, v2 = q2[0], q2[1:]
    return np.concatenate([[s1 * s2 - v1 @ v2], s1 * v2 + s2 * v1 + np.cross(v1, v2)])

def qconj(q):
    return np.concatenate([[q[0]], -q[1:]])

def quat_from_axis_angle(axis, theta):
    axis = axis / np.linalg.norm(axis)
    return np.concatenate([[np.cos(theta / 2)], np.sin(theta / 2) * axis])

def quat_rotate(q, v):
    return qmul(qmul(q, np.concatenate([[0.0], v])), qconj(q))[1:]

def quat_to_matrix(q):
    w, x, y, z = q
    return np.array([[1 - 2 * (y**2 + z**2), 2 * (x * y - w * z), 2 * (x * z + w * y)],
                     [2 * (x * y + w * z), 1 - 2 * (x**2 + z**2), 2 * (y * z - w * x)],
                     [2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x**2 + y**2)]])

# Hamilton 规则
qi, qj, qk = np.array([0, 1.0, 0, 0]), np.array([0, 0, 1.0, 0]), np.array([0, 0, 0, 1.0])
print(f'ij = {qmul(qi, qj)}, ji = {qmul(qj, qi)}, i^2 = {qmul(qi, qi)}, ijk = {qmul(qmul(qi, qj), qk)}')

axis, theta = np.array([1.0, 2.0, 2.0]), 1.3
v = np.array([0.5, -1.0, 3.0])
q = quat_from_axis_angle(axis, theta)
print(f'q v q* == Rodrigues ? {np.allclose(quat_rotate(q, v), rodrigues_vector(v, axis, theta))}')
print(f'q v q* 的标量部分 = {qmul(qmul(q, np.concatenate([[0.0], v])), qconj(q))[0]:.2e} (结果还是纯四元数)')
print(f'-q 和 q 是同一个旋转 ? {np.allclose(quat_rotate(-q, v), quat_rotate(q, v))}')

columns = np.column_stack([quat_rotate(q, e) for e in np.eye(3)])
print(f'矩阵公式 == 三个基向量的 q e q* ? {np.allclose(quat_to_matrix(q), columns)}')
# scipy 的四元数顺序是 [x, y, z, w] (标量在最后)
q_scipy = Rotation.from_rotvec(theta * axis / np.linalg.norm(axis)).as_quat()
print(f'scipy as_quat (x, y, z, w) = {q_scipy}, 我们的 (w, x, y, z) = {q}')
print(f'矩阵 == scipy ? {np.allclose(quat_to_matrix(q), Rotation.from_quat(np.roll(q, -1)).as_matrix())}')

q1 = quat_from_axis_angle(np.array([0, 0, 1.0]), 0.9)
q2 = quat_from_axis_angle(np.array([1.0, 1.0, 0]), -0.6)
print(f'先 q1 再 q2 == q2 q1 ? {np.allclose(quat_rotate(q2, quat_rotate(q1, v)), quat_rotate(qmul(q2, q1), v))}')
print(f'四元数乘法 <-> 矩阵乘法: R(q2 q1) == R(q2) R(q1) ? {np.allclose(quat_to_matrix(qmul(q2, q1)), quat_to_matrix(q2) @ quat_to_matrix(q1))}')

<a id="slerp"></a>

$$
\text{球面线性插值 (Slerp, Spherical Linear Interpolation): 在两个旋转之间平滑过渡} \\
\text{单位四元数是 4 维单位球面上的点, 两个旋转之间的最短路径是球面上的大圆弧} \\ \quad \\
\text{推导: } q_0, q_1 \text{ 的夹角 } \Omega \text{, } \cos\Omega = q_0 \cdot q_1 \text{ (4 维点积)} \\
\text{在 } q_0, q_1 \text{ 张成的平面里, 构造和 } q_0 \text{ 垂直的单位向量 (Gram-Schmidt):} \\
q_\perp = \frac{q_1 - \cos\Omega \, q_0}{\sin\Omega} \\
\text{沿大圆以匀角速度走, 角度 } t \Omega \text{: } q(t) = \cos(t\Omega) \, q_0 + \sin(t\Omega) \, q_\perp \\
= \cos(t\Omega) q_0 + \frac{\sin(t\Omega)}{\sin\Omega} q_1 - \frac{\sin(t\Omega)\cos\Omega}{\sin\Omega} q_0 \\
= \frac{\sin\Omega \cos(t\Omega) - \cos\Omega \sin(t\Omega)}{\sin\Omega} q_0 + \frac{\sin(t\Omega)}{\sin\Omega} q_1 \\
\text{分子用和角公式 } \sin(\Omega - t\Omega) = \sin\Omega\cos t\Omega - \cos\Omega \sin t\Omega \text{:} \\
\text{slerp}(q_0, q_1, t) = \frac{\sin((1 - t)\Omega)}{\sin\Omega} q_0 + \frac{\sin(t\Omega)}{\sin\Omega} q_1 \\ \quad \\
\text{注意: q 和 -q 是同一个旋转, 如果 } q_0 \cdot q_1 < 0 \text{, 先把 } q_1 \text{ 取反, 走较短的那一段弧} \\
\text{对比: 直接线性插值再归一化 (nlerp), 路径相同但角速度不均匀 (中间快两头慢)}
$$

数学基础: [Gram-Schmidt 正交化 (QR 分解)](../math_basics/linear_algebra.ipynb#qr)

In [ ]:
def slerp(q0, q1, t):
    q0, q1 = q0 / np.linalg.norm(q0), q1 / np.linalg.norm(q1)
    dot = q0 @ q1
    if dot < 0:                  # 走短弧
        q1, dot = -q1, -dot
    omega = np.arccos(np.clip(dot, -1, 1))
    t = np.atleast_1d(t)[:, None]
    return (np.sin((1 - t) * omega) * q0 + np.sin(t * omega) * q1) / np.sin(omega)

def nlerp(q0, q1, t):
    if q0 @ q1 < 0:
        q1 = -q1
    t = np.atleast_1d(t)[:, None]
    q = (1 - t) * q0 + t * q1
    return q / np.linalg.norm(q, axis=1, keepdims=True)

def rotation_angle_between(qa, qb):
    # 两个旋转之间的夹角 = 2 arccos(|qa . qb|)
    return 2 * np.arccos(np.clip(np.abs(np.sum(qa * qb, axis=-1)), -1, 1))

q0 = quat_from_axis_angle(np.array([0, 0, 1.0]), 0.0)
q1 = quat_from_axis_angle(np.array([1.0, 1.0, 1.0]), 2.5)
ts = np.linspace(0, 1, 11)
ours = slerp(q0, q1, ts)

key_rots = Rotation.from_quat(np.roll(np.vstack([q0, q1]), -1, axis=1))
scipy_slerp = Slerp([0, 1], key_rots)(ts).as_matrix()
print(f'slerp == scipy Slerp ? {np.allclose([quat_to_matrix(qq) for qq in ours], scipy_slerp)}')
print(f'slerp 结果都是单位四元数 ? {np.allclose(np.linalg.norm(ours, axis=1), 1)}')

ang_slerp = rotation_angle_between(q0, ours)
ang_nlerp = rotation_angle_between(q0, nlerp(q0, q1, ts))
print(f'slerp 每一步转过的角度 = {np.round(np.diff(ang_slerp), 4)} (均匀)')
print(f'nlerp 每一步转过的角度 = {np.round(np.diff(ang_nlerp), 4)} (中间大两头小)')

# 取反以后走短弧: q1 和 -q1 插值的中点表示同一个旋转
mid_a = slerp(q0, q1, 0.5)[0]
mid_b = slerp(q0, -q1, 0.5)[0]
print(f'slerp(q0, q1) 和 slerp(q0, -q1) 的中点是同一个旋转 ? {np.allclose(quat_to_matrix(mid_a), quat_to_matrix(mid_b))}')

plt.figure(figsize=(4, 3))
t_fine = np.linspace(0, 1, 200)
plt.plot(t_fine, rotation_angle_between(q0, slerp(q0, q1, t_fine)), label='slerp (constant speed)')
plt.plot(t_fine, rotation_angle_between(q0, nlerp(q0, q1, t_fine)), '--', label='nlerp')
plt.xlabel('t')
plt.ylabel('angle from q0 (rad)')
plot_nicely()
plt.show()